# Final Project — Education and Income Diversity in New York City (2023)

## Research question
Do NYC community districts with **higher educational attainment** tend to have **more equal** income distributions, or **more unequal** ones?

## Hypothesis
**H1:** Community districts with a higher share of residents who have a **Bachelor's degree or higher** will have a **higher Income Diversity Ratio (80:20)** — i.e., **more inequality**.

Why this hypothesis is plausible: higher-education districts can include both high-income earners and service-sector workers, which may widen the income distribution within the same geography.

# Data Sources
I use two datasets from the Citizens’ Committee for Children of New York (CCC) Data Portal, both derived from the U.S. Census Bureau American Community Survey (ACS) 1-Year Estimates. I first tried borough-level (dead end, n=5), then moved to community-district level after fixing messy geography labels and merging.

Dataset 1 — Educational Attainment (Adults Age 25+)

https://data.cccnewyork.org/data/table/1243/educational-attainment#1243/1419/131/a/a
Measures the distribution of education levels among adults age 25 and older.

Dataset 2 — Income Diversity Ratio (80:20 ratio)

https://data.cccnewyork.org/data/table/1423/income-diversity-ratio#1423/1701/131/a/a

The income diversity ratio measures inequality by dividing the income of the 80th percentile household by the income of the 20th percentile household.

# Import Libraries and Load Data

In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
edu = pd.read_csv("Educational Attainment.csv")
inc = pd.read_csv("Income Diversity Ratio.csv")
edu.head()
print(edu.shape, inc.shape)
print(edu.columns)
print(inc.columns)

(67, 7) (67, 7)
Index(['Rank / Location', 'Less than High School Degree', 'High School Degree',
       'Some College', 'Associate's Degree', 'Bachelor's Degree or Higher',
       'Unnamed: 6'],
      dtype='object')
Index(['Rank / Location', 'Income Diversity Ratio', 'Unnamed: 2', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6'],
      dtype='object')


# Data Cleaning — Educational Attainment

In [8]:
import pandas as pd
import numpy as np

edu = pd.read_csv("Educational Attainment.csv")

geo_col = "Rank / Location"
edu_cols = [
    "Less than High School Degree",
    "High School Degree",
    "Some College",
    "Associate's Degree",
    "Bachelor's Degree or Higher"
]

# drop section headers
edu = edu[~edu[geo_col].isin(["BOROUGHS", "COMMUNITY DISTRICTS"])]

# numeric
for c in edu_cols:
    edu[c] = (
        edu[c].astype(str)
        .str.replace(",", "", regex=False)
    )
    edu[c] = pd.to_numeric(edu[c], errors="coerce")

edu = edu.dropna(subset=edu_cols, how="all")
edu = edu[edu[geo_col].notna()]
edu[geo_col] = edu[geo_col].astype(str).str.strip()

# compute totals and share
edu["total_25plus"] = edu[edu_cols].sum(axis=1)
edu["bachelor_share"] = edu["Bachelor's Degree or Higher"] / edu["total_25plus"]

edu_clean = edu[[geo_col, "bachelor_share"]]

edu_clean.head()

,Rank / Location,bachelor_share
0,New York City,0.425279
2,Bronx,0.234972
3,Brooklyn,0.434637
4,Manhattan,0.649123
5,Queens,0.366257


# Data Cleaning — Income Diversity Ratio

In [9]:
import pandas as pd
import numpy as np

inc = pd.read_csv("Income Diversity Ratio.csv")

geo_col = "Rank / Location"
ratio_col = "Income Diversity Ratio"

# drop section headers
inc = inc[~inc[geo_col].isin(["BOROUGHS", "COMMUNITY DISTRICTS"])]

# numeric
inc[ratio_col] = pd.to_numeric(inc[ratio_col], errors="coerce")
inc = inc[inc[geo_col].notna()]
inc[geo_col] = inc[geo_col].astype(str).str.strip()

# remove rows without values
inc = inc.dropna(subset=[ratio_col])

inc_clean = inc[[geo_col, ratio_col]]

inc_clean.head()

,Rank / Location,Income Diversity Ratio
0,New York City,7.3
2,Bronx,7.4
3,Brooklyn,7.0
4,Manhattan,9.2
5,Queens,5.3


# Merge the Two Datasets
The two datasets share a geography label column (`Rank / Location`), but the formatting of community district names can differ across exports (extra spaces, parentheses, different borough naming conventions).

I will:
1) Do a **simple merge** using the raw geography label to see what we get.
2) Keep a **borough-only** view as a quick “sanity check” (but this is a dead end because n=5).
3) Try a **naive community-district merge** and quantify how many rows we lose (dead end).
4) Fix the merge by creating a standardized **community district key (`cd_key`)** and merging on that.

In [11]:
merged = pd.merge(
    edu_clean,
    inc_clean,
    on="Rank / Location",
    how="inner"
)

print("Merged rows (raw label merge):", merged.shape[0])
merged.head()


Merged rows (raw label merge): 65


,Rank / Location,bachelor_share,Income Diversity Ratio
0,New York City,0.425279,7.3
1,Bronx,0.234972,7.4
2,Brooklyn,0.434637,7.0
3,Manhattan,0.649123,9.2
4,Queens,0.366257,5.3


# Dead end 1 — Borough-only
A quick first look at boroughs is tempting, but it is not sufficient for the Final Project because we only have **5 data points**. I keep this step as part of my exploration, then move to the community-district level.

In [12]:
boroughs = ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]

merged_boro = merged[merged["Rank / Location"].isin(boroughs)].copy()
print("Borough rows (n):", merged_boro.shape[0])

merged_boro

Borough rows (n): 5


,Rank / Location,bachelor_share,Income Diversity Ratio
1,Bronx,0.234972,7.4
2,Brooklyn,0.434637,7.0
3,Manhattan,0.649123,9.2
4,Queens,0.366257,5.3
5,Staten Island,0.354702,5.6


# Dead end 2: Naive community-district merge on the raw label

Next I try merging **community districts** using the raw `Rank / Location` label.

This often fails because community-district names are not perfectly standardized between exports (e.g., spacing, parentheses, borough naming). I quantify the loss by comparing:
- education CD rows
- income CD rows
- rows that survive the naive merge

In [13]:
edu_cd_raw = edu_clean[~edu_clean["Rank / Location"].isin(boroughs)].copy()
inc_cd_raw = inc_clean[~inc_clean["Rank / Location"].isin(boroughs)].copy()

naive_cd = pd.merge(
    edu_cd_raw,
    inc_cd_raw,
    on="Rank / Location",
    how="inner"
)

print("Education community-district rows:", edu_cd_raw.shape[0])
print("Income community-district rows:", inc_cd_raw.shape[0])
print("Naive merged CD rows:", naive_cd.shape[0])

naive_cd.head()

Education community-district rows: 60
Income community-district rows: 60
Naive merged CD rows: 60


,Rank / Location,bachelor_share,Income Diversity Ratio
0,New York City,0.425279,7.3
1,Astoria\n(Q01),0.523441,5.2
2,Battery Park/Tribeca\n(M01),0.855127,3.5
3,Bay Ridge\n(K10),0.456544,5.6
4,Bayside\n(Q11),0.471444,5.1


# Fixing the merge: create a standardized community district key

To merge reliably, I extract a standardized key like:
- `MN01`, `BX12`, `BK03`, `QN05`, `SI01`

from the raw district label. Then I merge on `cd_key` instead of the raw string label.

This is the main merged dataset used for the rest of the analysis.

In [21]:
def extract_cd_key(text):
    t = str(text).upper().strip()
    t = t.replace("\n", " ")

    # (K10) (B07) (M02) (Q11) (S01)
    m = re.search(r"\(([BKQMS])\s*0?(\d{1,2})\)", t)
    if m:
        letter = m.group(1)
        num = int(m.group(2))
        borough_map = {"B":"BX","K":"BK","M":"MN","Q":"QN","S":"SI"}
        return f"{borough_map[letter]}{num:02d}"

    # fallback: K10 without parentheses
    m2 = re.search(r"\b([BKQMS])\s*0?(\d{1,2})\b", t)
    if m2:
        letter = m2.group(1)
        num = int(m2.group(2))
        borough_map = {"B":"BX","K":"BK","M":"MN","Q":"QN","S":"SI"}
        return f"{borough_map[letter]}{num:02d}"

    return np.nan

# Merge on `cd_key` and handle duplicates

Some exports may contain duplicate lines for the same district. To be safe, I average within each `cd_key` before merging.

In [22]:
edu_cd_fix = edu_cd_raw.copy()
inc_cd_fix = inc_cd_raw.copy()

edu_cd_fix["cd_key"] = edu_cd_fix["Rank / Location"].map(extract_cd_key)
inc_cd_fix["cd_key"] = inc_cd_fix["Rank / Location"].map(extract_cd_key)

print("Education coverage:", edu_cd_fix["cd_key"].notna().mean())
print("Income coverage:", inc_cd_fix["cd_key"].notna().mean())

edu_cd_final = (
    edu_cd_fix.dropna(subset=["cd_key","bachelor_share"])
    .groupby("cd_key", as_index=False)
    .agg(bachelor_share=("bachelor_share","mean"))
)

inc_cd_final = (
    inc_cd_fix.dropna(subset=["cd_key","Income Diversity Ratio"])
    .groupby("cd_key", as_index=False)
    .agg(income_diversity_ratio=("Income Diversity Ratio","mean"))
)

df = pd.merge(edu_cd_final, inc_cd_final, on="cd_key", how="inner")

print("Fixed merged CD rows:", df.shape[0])
df.head()
# Final analysis dataset
df = df.dropna(subset=["bachelor_share", "income_diversity_ratio"])

print("Final analysis observations:", df.shape[0])
df.describe()

Education coverage: 0.9833333333333333
Income coverage: 0.9833333333333333
Fixed merged CD rows: 59
Final analysis observations: 59


,bachelor_share,income_diversity_ratio
count,59.000000,59.000000
mean,0.419770,6.271186
std,0.199887,1.726872
min,0.110205,3.500000
25%,0.273770,5.100000
50%,0.384300,6.200000
75%,0.483930,6.850000
max,0.855127,11.900000


# Visualization 1: Education and Income Inequality

To examine the relationship between education and inequality, I visualize the association between the share of adults with a Bachelor's degree (education) and the Income Diversity Ratio (inequality) across all NYC community districts.

A scatter plot is appropriate because both variables are continuous. I also include a fitted trend line to evaluate the direction of the relationship.

In [23]:
import plotly.express as px

fig = px.scatter(
    df,
    x="bachelor_share",
    y="income_diversity_ratio",
    hover_name="cd_key",
    trendline="ols",
    title="Educational Attainment vs Income Inequality Across NYC Community Districts (2023)",
    labels={
        "bachelor_share": "Share of Adults with Bachelor's Degree or Higher",
        "income_diversity_ratio": "Income Diversity Ratio (80:20)"
    }
)

fig.update_layout(template="plotly_white")
fig.show()


# Interpretation of Scatter
The scatter plot does not show a strong positive relationship between educational attainment and income inequality. Instead, the fitted trend line slopes slightly downward, suggesting a weak negative association.

Community districts with higher shares of residents holding a Bachelor's degree or higher tend, on average, to have slightly lower income diversity ratios. However, the points are widely dispersed, indicating that the relationship is weak and education alone explains only a small portion of the variation in inequality across neighborhoods.

Several districts with moderate education levels display very high inequality, while some of the most highly educated districts show relatively moderate inequality. This indicates that factors other than education — such as housing markets, occupational mix, and neighborhood composition — likely play a larger role in shaping income inequality.

Overall, the scatter plot provides little visual evidence that higher education levels systematically increase inequality, and if anything, the relationship appears weakly negative rather than positive.

# Why Borough-Level Analysis Is Not Sufficient

New York City has only five boroughs. Analyzing inequality at the borough level would produce only five observations, which is insufficient for meaningful statistical analysis and would hide large variation within boroughs.

Community districts provide a much more appropriate unit of analysis because they better represent neighborhood-level socioeconomic structure. For example, Manhattan includes both very wealthy and relatively lower-income neighborhoods. Borough averages would obscure this internal inequality.

In [24]:
from scipy.stats import pearsonr, spearmanr

pearson_corr, pearson_p = pearsonr(df["bachelor_share"], df["income_diversity_ratio"])
spearman_corr, spearman_p = spearmanr(df["bachelor_share"], df["income_diversity_ratio"])

print("Pearson correlation:", pearson_corr)
print("Pearson p-value:", pearson_p)

print("Spearman correlation:", spearman_corr)
print("Spearman p-value:", spearman_p)

Pearson correlation: -0.2964161774349502
Pearson p-value: 0.022631699447915998
Spearman correlation: -0.27395825423716585
Spearman p-value: 0.0357603089999686


# Statistical Interpretation
Both Pearson and Spearman correlation tests indicate a negative association between educational attainment and income inequality.

The Pearson correlation coefficient is approximately −0.30 and statistically significant (p < 0.05), meaning that community districts with higher shares of residents holding a Bachelor's degree tend to have lower income diversity ratios. The Spearman rank correlation shows a similar negative relationship, confirming that the pattern is not driven by a few extreme observations.

Because both correlations are statistically significant, the observed relationship is unlikely to be due to random variation.

Therefore, the results do not support the original hypothesis. Instead of higher education being associated with higher inequality, the data suggests that more educated neighborhoods in New York City tend to have slightly more equal income distributions on average.

However, the magnitude of the correlation is moderate rather than strong, indicating that educational attainment explains only part of the variation in neighborhood inequality.

# Visualization 2: Quintile Analysis
To better understand the relationship, I grouped community districts into five equal-sized groups (quintiles) based on educational attainment.

In [25]:
# create education quintiles
df["edu_quintile"] = pd.qcut(df["bachelor_share"], 5, labels=False)

quintile_means = (
    df.groupby("edu_quintile")["income_diversity_ratio"]
    .mean()
    .reset_index()
)

fig2 = px.line(
    quintile_means,
    x="edu_quintile",
    y="income_diversity_ratio",
    markers=True,
    title="Average Income Inequality by Education Quintile",
    labels={
        "edu_quintile": "Education Quintile (Lowest → Highest)",
        "income_diversity_ratio": "Average Income Diversity Ratio"
    }
)

fig2.update_layout(template="plotly_white")
fig2.show()

# Quintile Comparison

The quintile plot shows that the average income diversity ratio generally decreases as educational attainment increases. Districts in the lowest education quintile have the highest average inequality, while districts in the highest education quintile have the lowest inequality.

Although the pattern is not perfectly monotonic, the overall trend is clearly downward. This supports the negative correlation observed in the scatter plot and statistical tests.

This suggests that neighborhoods with higher shares of college-educated residents tend, on average, to have somewhat more equal income distributions. One possible explanation is that highly educated districts may have more uniformly middle-to-upper income populations, whereas less educated districts may contain a wider spread between very low-income households and moderate-income households.

Therefore, the quintile analysis reinforces the conclusion that higher educational attainment is associated with slightly lower measured inequality at the community district level.

# Conclusion
This project examined whether higher educational attainment in New York City community districts is associated with greater income inequality. The original hypothesis predicted a positive relationship, expecting that highly educated neighborhoods would contain both high-income professionals and low-wage service workers, thereby widening the income distribution.

The results do not support this hypothesis. The scatter plot shows a weak downward trend, and both Pearson and Spearman correlation tests indicate a statistically significant negative relationship between the share of residents with a Bachelor's degree and the Income Diversity Ratio. The quintile analysis further confirms this pattern: districts with higher educational attainment tend to have slightly lower levels of measured inequality on average.

These findings suggest that, at the neighborhood level in New York City, education is associated with somewhat more equal income distributions rather than greater inequality. One possible explanation is that highly educated neighborhoods are economically more homogeneous, with a larger concentration of middle-to-upper income households, while lower-education districts may include a wider range of low- and moderate-income households.

However, the magnitude of the relationship is modest. Educational attainment explains only part of the variation in income inequality across community districts. Other factors — including housing prices, labor market structure, demographic composition, and local economic conditions — likely play a more substantial role.

Overall, the analysis demonstrates that education alone does not increase neighborhood inequality and may instead be associated with modestly greater income equality.

# Limitations and AI Disclosure
This project used publicly available data from the Citizens’ Committee for Children of New York (CCC) Data Portal, derived from the U.S. Census Bureau American Community Survey.

This analysis uses cross-sectional data for a single year (2023), so it cannot establish causality. The relationship observed is correlational rather than causal. Additionally, the Income Diversity Ratio measures dispersion but does not identify the mechanisms producing inequality.

Generative AI tools were used only for language polishing and debugging assistance. All data selection, analysis decisions, visualizations, and interpretations were conducted by myself.